# Notebook #6: Citation Descriptives
We begin by generating a summary table, comparing the citation statistics between *sharing* and *non-sharing* articles, as well as a drill-down to the sharing subcategories (granularities). We compare both the citation counts and the FWCI scores, providing the median, mean and standard deviation for each group. We also calculate the group-level h-index for each sharing category.<br>
The citation count is the cumulative number of citations through the latest calendar year that is complete for every article (`dataset.citations_through_shared_year`), the same quantity the citation models use as their outcome, and not OpenAlex's `TotalCitations`, whose cutoff differs between articles. The h-index is computed from these counts.

In [ ]:
import pandas as pd

from helpers import dataset

In [3]:
def calculate_h_index(citations: pd.Series) -> int:
    """Calculates h-index from a Series of citation counts."""
    if citations.empty:
        return 0
    # Sort descending (highest citations first)
    sorted_cits = sorted(citations.dropna().astype(int), reverse=True)
    # Count how many papers have citations >= their rank
    return sum(c >= i + 1 for i, c in enumerate(sorted_cits))

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Summary Table

In [ ]:
citations, SHARED_YEAR = dataset.citations_through_shared_year(combined)
CITATIONS = f"Citations through {SHARED_YEAR}"
impact = combined.assign(**{CITATIONS: citations})

_IMPACT_AGGS = {
    CITATIONS: ["count", calculate_h_index, "median", "mean", "std"],
    "FieldWeightedCitationIndex": ["count", "median", "mean", "std"],
}

metrics_per_sharing_class = (
    impact[["data_sharing_class", CITATIONS, "FieldWeightedCitationIndex"]]
    .groupby("data_sharing_class")
    .agg(_IMPACT_AGGS)
)
metrics_for_any_data_sharing = (
    impact.loc[impact["is_sharing_data"], [CITATIONS, "FieldWeightedCitationIndex"]]
    .agg(_IMPACT_AGGS)
)

metrics_for_data_sharing = (
    pd.concat([
        metrics_per_sharing_class,
        metrics_for_any_data_sharing.stack().rename("SHARE").reorder_levels([1, 0]).to_frame().T
    ])
    .dropna(how="all", axis=1)
    .drop(columns=[("FieldWeightedCitationIndex", "count")])
    .rename(columns={"count": "N", "calculate_h_index": "H-Index"}, level=1)
)
metrics_for_data_sharing